# Build Groups E and F

Make a "Group E" and "Group F".  Using the cleaned data. Both groups should have an initial score in Initial 2022, they should have summative tests in 2022, 2023, 2024 with dates after the initial test. They should have enrollment data for all three years. Group E should have an initial assessment of Novice Learner, and Group F should have Intermediate Learner. Also all summative scores 1150

In [2]:
import pandas as pd
import os

# --- Paths ---
elpac_clean_dir = "../1095 Data/ELPAC/Cleaned"
enrollment_dir  = "../1095 Data/Enrollment"

# --- Load Cleaned Initial 2022 ---
initial_22 = pd.read_csv(os.path.join(elpac_clean_dir, "InitialElpac_22_clean.csv"), low_memory=False)
initial_22 = initial_22.apply(lambda col: col.str.strip() if col.dtype == "object" else col)
initial_22["OverallPL"]      = initial_22["OverallPL"].astype(str).str.strip()
initial_22["TestStartDate"]  = pd.to_datetime(initial_22["TestStartDate"], errors="coerce")
initial_22["OverallScaleScore"] = pd.to_numeric(initial_22["OverallScaleScore"], errors="coerce")
print(f"Initial 2022 (clean): {len(initial_22):,} rows")
print(f"Unique OverallPL values:")
for val, count in initial_22["OverallPL"].value_counts().items():
    print(f"  '{val}': {count:,}")

# --- Load Cleaned Summative files ---
summative_files = {
    2022: "SummativeElpac_22_clean.csv",
    2023: "SummativeElpac_23_clean.csv",
    2024: "SummativeElpac_24_clean.csv",
}

summative_dfs = {}
for year, filename in summative_files.items():
    df = pd.read_csv(os.path.join(elpac_clean_dir, filename), low_memory=False)
    df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)
    df["OverallScaleScore"] = pd.to_numeric(df["OverallScaleScore"], errors="coerce")
    df["TestStartDate"]     = pd.to_datetime(df["TestStartDate"], errors="coerce")
    summative_dfs[year]     = df
    print(f"Summative {year} (clean): {len(df):,} rows")

# --- Load Enrollment files ---
enrollment_files = {
    "2021-22": "CEnrollment_21-22.csv",
    "2022-23": "CEnrollment_22-23.csv",
    "2023-24": "CEnrollment_23-24.csv",
}

enrollment_dfs = {}
for label, filename in enrollment_files.items():
    enrollment_dfs[label] = pd.read_csv(os.path.join(enrollment_dir, filename), low_memory=False)
    print(f"Enrollment {label}: {len(enrollment_dfs[label]):,} rows")

# --- SSIDs present in all 3 enrollment years ---
ssids_enroll_2122 = set(enrollment_dfs["2021-22"]["SSID"].dropna())
ssids_enroll_2223 = set(enrollment_dfs["2022-23"]["SSID"].dropna())
ssids_enroll_2324 = set(enrollment_dfs["2023-24"]["SSID"].dropna())
ssids_all_enrollment = ssids_enroll_2122 & ssids_enroll_2223 & ssids_enroll_2324
print(f"\nSSIDs with enrollment in all 3 years: {len(ssids_all_enrollment):,}")

# --- Build summative lookups: one record per student per year (already deduplicated in cleaned files) ---
summative_dates  = {}
summative_scores = {}
for year, df in summative_dfs.items():
    valid = df[["SSID", "TestStartDate", "OverallScaleScore"]].dropna()
    summative_dates[year]  = dict(zip(valid["SSID"], valid["TestStartDate"]))
    summative_scores[year] = dict(zip(valid["SSID"], valid["OverallScaleScore"]))

# --- Build initial lookup: SSID -> {date, pl} ---
initial_lookup = {}
for _, row in initial_22[["SSID", "TestStartDate", "OverallPL"]].dropna().iterrows():
    initial_lookup[row["SSID"]] = {
        "date": row["TestStartDate"],
        "pl":   row["OverallPL"]
    }
print(f"Students in Initial 2022: {len(initial_lookup):,}")

# --- Build Group function ---
def build_group(target_pl, min_score=1150):
    group_ssids     = []
    no_enrollment   = 0
    no_summative    = 0
    bad_date        = 0
    bad_score       = 0

    for ssid, init in initial_lookup.items():
        if init["pl"] != target_pl:
            continue

        # Must have enrollment in all 3 years
        if ssid not in ssids_all_enrollment:
            no_enrollment += 1
            continue

        init_date = init["date"]
        if pd.isna(init_date):
            bad_date += 1
            continue

        # Must have summative in all 3 years
        if not all(ssid in summative_dates[year] for year in [2022, 2023, 2024]):
            no_summative += 1
            continue

        # All summative dates must be after initial date
        if not all(summative_dates[year][ssid] > init_date for year in [2022, 2023, 2024]):
            bad_date += 1
            continue

        # All summative scores must be >= min_score
        if not all(summative_scores[year][ssid] >= min_score for year in [2022, 2023, 2024]):
            bad_score += 1
            continue

        group_ssids.append(ssid)

    print(f"\n  '{target_pl}':")
    print(f"    Passed all filters:         {len(group_ssids):,}")
    print(f"    Excluded - no enrollment:   {no_enrollment:,}")
    print(f"    Excluded - no summative:    {no_summative:,}")
    print(f"    Excluded - date issue:      {bad_date:,}")
    print(f"    Excluded - score < {min_score}: {bad_score:,}")
    return group_ssids

# --- Build Groups E and F ---
print("\nBuilding groups...")
group_e_ssids = build_group("Novice English Learner")
group_f_ssids = build_group("Intermediate English Learner")

print(f"\nGroup E (Novice):       {len(group_e_ssids):,} students")
print(f"Group F (Intermediate): {len(group_f_ssids):,} students")

# --- Verify: spot check a sample student from each group ---
for group_name, group_ssids in [("Group E", group_e_ssids), ("Group F", group_f_ssids)]:
    if len(group_ssids) == 0:
        print(f"\n{group_name}: EMPTY — no sample to check")
        continue
    sample = group_ssids[0]
    print(f"\n{group_name} sample SSID: {sample}")
    print(f"  Initial PL:    {initial_lookup[sample]['pl']}")
    print(f"  Initial date:  {initial_lookup[sample]['date']}")
    for year in [2022, 2023, 2024]:
        print(f"  Summative {year}: date={summative_dates[year].get(sample)}, "
              f"score={summative_scores[year].get(sample)}")

# --- Export to CSV ---
pd.DataFrame({"SSID": group_e_ssids}).to_csv("group_e_ssids.csv", index=False)
pd.DataFrame({"SSID": group_f_ssids}).to_csv("group_f_ssids.csv", index=False)
print("\nSaved: group_e_ssids.csv")
print("Saved: group_f_ssids.csv")

Initial 2022 (clean): 62,946 rows
Unique OverallPL values:
  'Novice English Learner': 45,541
  'Intermediate English Learner': 8,799
  'Initial Fluent English Proficient (IFEP)': 7,262
  '': 936
  'NS': 408
Summative 2022 (clean): 302,476 rows
Summative 2023 (clean): 296,681 rows
Summative 2024 (clean): 291,920 rows
Enrollment 2021-22: 1,519,618 rows
Enrollment 2022-23: 1,467,738 rows
Enrollment 2023-24: 1,471,765 rows

SSIDs with enrollment in all 3 years: 988,063
Students in Initial 2022: 61,823

Building groups...

  'Novice English Learner':
    Passed all filters:         28,707
    Excluded - no enrollment:   9,128
    Excluded - no summative:    7,216
    Excluded - date issue:      0
    Excluded - score < 1150: 367

  'Intermediate English Learner':
    Passed all filters:         4,196
    Excluded - no enrollment:   1,476
    Excluded - no summative:    3,125
    Excluded - date issue:      0
    Excluded - score < 1150: 2

Group E (Novice):       28,707 students
Group F (I